# Lecture 13 · nanoGPT from scratch

Build a working character-level GPT in ~100 lines · train on Tiny Shakespeare · generate text.

**Based on** · Karpathy's nanoGPT and *Let's build GPT* video.

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(1337)

## 1 · Load Tiny Shakespeare

A 1 MB file · every play Shakespeare wrote, concatenated.

In [ ]:
# Download if needed
import os, urllib.request
if not os.path.exists('tinyshakespeare.txt'):
    url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
    urllib.request.urlretrieve(url, 'tinyshakespeare.txt')

with open('tinyshakespeare.txt') as f:
    text = f.read()

print(f'chars: {len(text):,}')
print(f'sample:\n{text[:400]}')

# Char-level tokenizer
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for i, c in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda idx: ''.join(itos[i] for i in idx)

print(f'\nvocab size: {vocab_size}')
print(f'vocab: {"".join(chars)}')

In [ ]:
# Train/val split
data = torch.tensor(encode(text), dtype=torch.long)
split = int(0.9 * len(data))
train_data = data[:split]
val_data = data[split:]
print(f'train: {len(train_data):,}   val: {len(val_data):,}')

## 2 · The Transformer block

Pre-norm, multi-head self-attention, FFN, residuals · exactly the L13 diagram.

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, block_size):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.proj = nn.Linear(d_model, d_model)
        self.register_buffer('mask',
            torch.tril(torch.ones(block_size, block_size)).view(1, 1, block_size, block_size))

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x)  # B, T, 3C
        q, k, v = qkv.split(C, dim=-1)
        q = q.view(B, T, self.n_heads, self.d_head).transpose(1, 2)  # B, H, T, d_head
        k = k.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_head).transpose(1, 2)

        # Scaled dot-product attention with causal mask
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        out = att @ v  # B, H, T, d_head
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(out)


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, block_size):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, block_size)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model),
        )

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x


class nanoGPT(nn.Module):
    def __init__(self, vocab_size, d_model=192, n_heads=6, n_layers=6, block_size=128):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(block_size, d_model)
        self.blocks = nn.ModuleList([TransformerBlock(d_model, n_heads, block_size)
                                      for _ in range(n_layers)])
        self.norm_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.tok_emb(idx) + self.pos_emb(pos)
        for block in self.blocks:
            x = block(x)
        x = self.norm_f(x)
        logits = self.head(x)

        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
            return logits, loss
        return logits, None

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = float('-inf')
            probs = F.softmax(logits, dim=-1)
            next_tok = torch.multinomial(probs, 1)
            idx = torch.cat([idx, next_tok], dim=1)
        return idx

model = nanoGPT(vocab_size).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'nanoGPT · {n_params/1e6:.2f}M params')

## 3 · Training loop

~5 minutes on a GPU · 30 minutes on CPU.

In [ ]:
batch_size = 32
block_size = 128
steps = 3000
lr = 3e-4

def get_batch(data):
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

opt = torch.optim.AdamW(model.parameters(), lr=lr)

for step in range(steps):
    x, y = get_batch(train_data)
    _, loss = model(x, y)
    opt.zero_grad(); loss.backward(); opt.step()

    if step % 200 == 0:
        # Quick val check
        with torch.no_grad():
            xv, yv = get_batch(val_data)
            _, val_loss = model(xv, yv)
        print(f'step {step:4d}  train {loss.item():.3f}  val {val_loss.item():.3f}')

## 4 · Generate Shakespeare

In [ ]:
# Prompt = a single newline
prompt = torch.tensor([[encode('\n')[0]]], device=device)
out = model.generate(prompt, max_new_tokens=500, temperature=0.8, top_k=50)
print(decode(out[0].tolist()))

## 5 · What to try next

- **Tune hyperparameters** · layers 6→12, d_model 192→384. Watch params grow.
- **Longer context** · `block_size = 256`. Needs more memory.
- **Sampling** · play with temperature (0.2 = boring, 1.5 = chaotic).
- **New dataset** · train on your own text.
- **Karpathy's video** · watch *Let's build GPT: from scratch, in code, spelled out* for the full derivation.

**Reflection** · this is the same architecture as GPT-2, GPT-3, Llama — scaled down 1000× and trained for minutes. The code above literally is the recipe, just with smaller numbers.